# Chapter 7.2: RL in Production Systems

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand YouTube's REINFORCE-based recommendation system architecture
2. Implement off-policy corrected REINFORCE for ranking
3. Design safe exploration strategies for production systems
4. Implement reward shaping that combines short-term and long-term signals
5. Handle delayed rewards and credit assignment in recommendation
6. Apply batch RL techniques to learn from logged interaction data
7. Evaluate production RL systems with A/B testing considerations

## Prerequisites

- Chapter 7.1 (RL fundamentals for recommendation)
- Understanding of policy gradient methods (REINFORCE)
- Familiarity with production ML system constraints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part7/chapter_7.2_rl_production.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part7/chapter_7.2_rl_production.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque, defaultdict
import random
from typing import List, Tuple, Dict, Optional

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. YouTube's REINFORCE System

YouTube's recommendation system (Chen et al., 2019 — "Top-K Off-Policy Correction for a REINFORCE Recommender System") uses REINFORCE with off-policy correction to optimize for long-term user satisfaction.

### Key Architecture

1. **Logging policy** $\beta$: The current production model that collects data
2. **Target policy** $\pi_\theta$: The new policy being trained
3. **Off-policy correction**: Importance weights $w = \frac{\pi_\theta(a|s)}{\beta(a|s)}$ correct for distribution mismatch

### Off-Policy REINFORCE

$$\nabla_\theta J(\theta) = \mathbb{E}_{\beta}\left[\frac{\pi_\theta(a_t|s_t)}{\beta(a_t|s_t)} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R_t\right]$$

With top-K correction for recommending a slate of K items:

$$w_K(a|s) = \frac{\pi_\theta(a|s)}{\max\left(\frac{1}{K}, \beta(a|s)\right)}$$

> **💡 Concept:** The $\max(1/K, \beta)$ denominator prevents importance weights from exploding when the logging policy rarely selects certain items. This is critical for production stability.

In [ ]:
class ProductionRecEnv:
    """Simulates a production recommendation environment with multiple signals."""
    
    def __init__(self, n_users: int = 500, n_items: int = 100,
                 embed_dim: int = 16, seed: int = 42):
        self.rng = np.random.RandomState(seed)
        self.n_users = n_users
        self.n_items = n_items
        self.embed_dim = embed_dim
        
        # User and item embeddings
        self.user_embeds = self.rng.randn(n_users, embed_dim) * 0.3
        self.item_embeds = self.rng.randn(n_items, embed_dim) * 0.3
        
        # Item properties
        self.item_quality = self.rng.beta(2, 3, n_items)  # Content quality
        self.item_clickbait = self.rng.beta(3, 2, n_items)  # Clickbait score
        self.item_duration = self.rng.exponential(5, n_items)  # Avg watch time (min)
    
    def get_user_state(self, user_id: int, history: List[int]) -> np.ndarray:
        user_embed = self.user_embeds[user_id]
        if len(history) > 0:
            hist_embed = self.item_embeds[history[-5:]].mean(axis=0)
        else:
            hist_embed = np.zeros(self.embed_dim)
        return np.concatenate([user_embed, hist_embed]).astype(np.float32)
    
    def get_rewards(self, user_id: int, item_id: int) -> Dict[str, float]:
        """Return multiple reward signals."""
        similarity = np.dot(self.user_embeds[user_id], self.item_embeds[item_id])
        
        # Click probability: influenced by clickbait + relevance
        click_logit = similarity * 2 + self.item_clickbait[item_id] * 1.5
        click_prob = 1.0 / (1.0 + np.exp(-click_logit))
        clicked = float(self.rng.random() < click_prob)
        
        # Watch time: only if clicked, depends on quality
        watch_time = 0.0
        if clicked:
            base_time = self.item_duration[item_id] * self.item_quality[item_id]
            watch_time = max(0, base_time + self.rng.normal(0, 1))
        
        # Satisfaction: quality-weighted engagement
        satisfaction = watch_time * self.item_quality[item_id] * 0.1
        
        return {
            'click': clicked,
            'watch_time': watch_time,
            'satisfaction': satisfaction,
            'click_prob': click_prob
        }

env = ProductionRecEnv(seed=42)
print(f"Environment: {env.n_users} users, {env.n_items} items")

In [ ]:
class LoggingPolicy(nn.Module):
    """Simulated logging (production) policy."""
    
    def __init__(self, state_dim: int, n_items: int, temperature: float = 1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_items)
        )
        self.temperature = temperature
        # Pre-train with random weights (simulates existing production model)
        with torch.no_grad():
            for p in self.parameters():
                p.data = p.data * 0.5
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        logits = self.net(state) / self.temperature
        return F.softmax(logits, dim=-1)


class OffPolicyREINFORCE(nn.Module):
    """Off-policy corrected REINFORCE for recommendation (YouTube-style)."""
    
    def __init__(self, state_dim: int, n_items: int, hidden_dim: int = 64):
        super().__init__()
        self.policy_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_items)
        )
        self.n_items = n_items
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        logits = self.policy_net(state)
        return F.softmax(logits, dim=-1)
    
    def get_log_prob(self, state: torch.Tensor, action: int) -> torch.Tensor:
        probs = self.forward(state)
        return torch.log(probs[action] + 1e-10)


def collect_logged_data(env, logging_policy, n_episodes: int = 500,
                        episode_len: int = 10) -> List[Dict]:
    """Collect data from logging policy."""
    logged_data = []
    
    for _ in range(n_episodes):
        user_id = np.random.randint(env.n_users)
        history = list(np.random.choice(env.n_items, 3, replace=False))
        
        for t in range(episode_len):
            state = env.get_user_state(user_id, history)
            state_t = torch.FloatTensor(state)
            
            with torch.no_grad():
                probs = logging_policy(state_t).numpy()
            
            action = np.random.choice(env.n_items, p=probs)
            rewards = env.get_rewards(user_id, action)
            
            logged_data.append({
                'state': state,
                'action': action,
                'logging_prob': float(probs[action]),
                'rewards': rewards,
                'user_id': user_id,
                'timestep': t
            })
            
            if rewards['click'] > 0:
                history.append(action)
    
    return logged_data


state_dim = env.embed_dim * 2
logging_policy = LoggingPolicy(state_dim, env.n_items)
logged_data = collect_logged_data(env, logging_policy, n_episodes=500)
print(f"Collected {len(logged_data)} logged interactions")
print(f"Average click rate: {np.mean([d['rewards']['click'] for d in logged_data]):.3f}")
print(f"Average watch time: {np.mean([d['rewards']['watch_time'] for d in logged_data]):.3f}")

In [ ]:
def train_off_policy_reinforce(logged_data, target_policy, optimizer,
                                 n_epochs: int = 30, batch_size: int = 128,
                                 top_k: int = 10, gamma: float = 0.99,
                                 reward_type: str = 'satisfaction',
                                 weight_clip: float = 10.0):
    """Train off-policy REINFORCE with top-K correction."""
    losses = []
    
    for epoch in range(n_epochs):
        random.shuffle(logged_data)
        epoch_loss = 0.0
        n_batches = 0
        
        for i in range(0, len(logged_data), batch_size):
            batch = logged_data[i:i+batch_size]
            
            states = torch.FloatTensor(np.array([d['state'] for d in batch]))
            actions = [d['action'] for d in batch]
            rewards = torch.FloatTensor([d['rewards'][reward_type] for d in batch])
            logging_probs = torch.FloatTensor([d['logging_prob'] for d in batch])
            
            # Target policy probabilities
            target_probs = target_policy(states)
            target_action_probs = target_probs[range(len(batch)), actions]
            
            # Top-K importance weights
            denominator = torch.max(torch.tensor(1.0 / top_k), logging_probs)
            weights = target_action_probs.detach() / denominator
            weights = torch.clamp(weights, 0, weight_clip)
            
            # Policy gradient loss with off-policy correction
            log_probs = torch.log(target_action_probs + 1e-10)
            loss = -torch.mean(weights * log_probs * rewards)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(target_policy.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / max(n_batches, 1)
        losses.append(avg_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")
    
    return losses


# Train policies optimizing for different rewards
results = {}

for reward_type in ['click', 'satisfaction']:
    print(f"\n--- Training with reward: {reward_type} ---")
    policy = OffPolicyREINFORCE(state_dim, env.n_items)
    opt = optim.Adam(policy.parameters(), lr=1e-3)
    losses = train_off_policy_reinforce(logged_data, policy, opt,
                                         n_epochs=30, reward_type=reward_type)
    results[reward_type] = {'policy': policy, 'losses': losses}

# Plot training curves
fig, ax = plt.subplots(figsize=(8, 4))
for name, data in results.items():
    ax.plot(data['losses'], label=f'Optimize: {name}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Off-Policy REINFORCE Training (Different Reward Signals)')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Safe Exploration in Production

Unrestricted exploration in production can degrade user experience. Safe exploration strategies include:

1. **Constrained exploration**: Limit deviation from the logging policy
2. **Conservative policy improvement**: Only update if improvement is guaranteed
3. **Exploration budget**: Allocate a fraction of traffic to exploration

$$\pi_{\text{safe}}(a|s) = (1-\alpha) \cdot \beta(a|s) + \alpha \cdot \pi_{\text{explore}}(a|s)$$

where $\alpha$ is the exploration budget.

> **⚠️ Common Pitfall:** Even small exploration rates can serve millions of "bad" recommendations per day at scale. Always set guardrails on minimum item quality and user-level exploration limits.

In [ ]:
class SafeExplorationPolicy:
    """Policy that blends exploitation with safe exploration."""
    
    def __init__(self, base_policy, exploration_policy, 
                 alpha: float = 0.1, min_quality: float = 0.3):
        self.base_policy = base_policy
        self.exploration_policy = exploration_policy
        self.alpha = alpha
        self.min_quality = min_quality  # Minimum item quality threshold
    
    def select_action(self, state: np.ndarray, item_qualities: np.ndarray) -> int:
        """Select action with safe exploration."""
        state_t = torch.FloatTensor(state)
        
        # Quality mask: only explore among acceptable items
        quality_mask = item_qualities >= self.min_quality
        
        if np.random.random() < self.alpha and quality_mask.sum() > 0:
            # Explore: sample from exploration policy, filtered by quality
            with torch.no_grad():
                explore_probs = self.exploration_policy(state_t).numpy()
            explore_probs[~quality_mask] = 0
            if explore_probs.sum() > 0:
                explore_probs /= explore_probs.sum()
                return np.random.choice(len(explore_probs), p=explore_probs)
        
        # Exploit: use base policy
        with torch.no_grad():
            base_probs = self.base_policy(state_t).numpy()
        return np.random.choice(len(base_probs), p=base_probs)


# Compare exploration strategies
def evaluate_policy(env, policy_fn, n_episodes: int = 200, episode_len: int = 10):
    """Evaluate a policy by running episodes."""
    all_clicks = []
    all_satisfaction = []
    
    for _ in range(n_episodes):
        user_id = np.random.randint(env.n_users)
        history = list(np.random.choice(env.n_items, 3, replace=False))
        
        for t in range(episode_len):
            state = env.get_user_state(user_id, history)
            action = policy_fn(state)
            rewards = env.get_rewards(user_id, action)
            all_clicks.append(rewards['click'])
            all_satisfaction.append(rewards['satisfaction'])
            if rewards['click'] > 0:
                history.append(action)
    
    return {'click_rate': np.mean(all_clicks), 'avg_satisfaction': np.mean(all_satisfaction)}


# Evaluate different exploration budgets
alphas = [0.0, 0.05, 0.1, 0.2, 0.5]
click_rates = []
satisfactions = []

base_pol = results['satisfaction']['policy']
explore_pol = LoggingPolicy(state_dim, env.n_items, temperature=2.0)  # Higher temp = more uniform

for alpha in alphas:
    safe_pol = SafeExplorationPolicy(base_pol, explore_pol, alpha=alpha)
    
    def policy_fn(state, sp=safe_pol):
        return sp.select_action(state, env.item_quality)
    
    metrics = evaluate_policy(env, policy_fn, n_episodes=200)
    click_rates.append(metrics['click_rate'])
    satisfactions.append(metrics['avg_satisfaction'])
    print(f"Alpha={alpha:.2f} | CTR: {metrics['click_rate']:.4f} | Satisfaction: {metrics['avg_satisfaction']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(alphas)), click_rates, tick_label=[str(a) for a in alphas], color='steelblue')
axes[0].set_xlabel('Exploration Budget (alpha)')
axes[0].set_ylabel('Click-Through Rate')
axes[0].set_title('CTR vs Exploration Budget')

axes[1].bar(range(len(alphas)), satisfactions, tick_label=[str(a) for a in alphas], color='coral')
axes[1].set_xlabel('Exploration Budget (alpha)')
axes[1].set_ylabel('Average Satisfaction')
axes[1].set_title('Satisfaction vs Exploration Budget')

plt.tight_layout()
plt.show()

## 3. Reward Shaping: Short-Term vs Long-Term

Production systems must balance multiple objectives. A common approach is to combine rewards:

$$r_{\text{combined}} = w_1 \cdot \text{click} + w_2 \cdot \text{watch\_time} + w_3 \cdot \text{satisfaction} - w_4 \cdot \text{regret}$$

> **🔑 Pro Tip:** In YouTube's system, they found that optimizing purely for clicks led to clickbait, while optimizing purely for watch time led to long but low-quality content. The key is finding the right balance through reward shaping.

In [ ]:
class RewardShaper:
    """Combines multiple reward signals into a single shaped reward."""
    
    def __init__(self, weights: Dict[str, float]):
        self.weights = weights
    
    def shape(self, rewards: Dict[str, float]) -> float:
        total = 0.0
        for key, weight in self.weights.items():
            if key in rewards:
                total += weight * rewards[key]
        return total


# Compare different reward shaping strategies
shaping_configs = {
    'Click Only': {'click': 1.0},
    'Watch Time': {'watch_time': 1.0},
    'Satisfaction': {'satisfaction': 1.0},
    'Balanced': {'click': 0.3, 'watch_time': 0.3, 'satisfaction': 0.4},
}

shaping_results = {}
for name, weights in shaping_configs.items():
    shaper = RewardShaper(weights)
    
    # Reshape logged data rewards
    shaped_data = []
    for d in logged_data:
        shaped_d = dict(d)
        shaped_d['rewards'] = dict(d['rewards'])
        shaped_d['rewards']['shaped'] = shaper.shape(d['rewards'])
        shaped_data.append(shaped_d)
    
    # Train policy
    policy = OffPolicyREINFORCE(state_dim, env.n_items)
    opt = optim.Adam(policy.parameters(), lr=1e-3)
    losses = train_off_policy_reinforce(shaped_data, policy, opt,
                                         n_epochs=20, reward_type='shaped')
    
    # Evaluate
    def policy_fn(state, p=policy):
        with torch.no_grad():
            probs = p(torch.FloatTensor(state)).numpy()
        return np.random.choice(env.n_items, p=probs)
    
    metrics = evaluate_policy(env, policy_fn, n_episodes=200)
    shaping_results[name] = metrics
    print(f"{name:20s} | CTR: {metrics['click_rate']:.4f} | Satisfaction: {metrics['avg_satisfaction']:.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
names = list(shaping_results.keys())
ctrs = [shaping_results[n]['click_rate'] for n in names]
sats = [shaping_results[n]['avg_satisfaction'] for n in names]

ax.scatter(ctrs, sats, s=150, zorder=5)
for i, name in enumerate(names):
    ax.annotate(name, (ctrs[i], sats[i]), textcoords="offset points",
                xytext=(10, 5), fontsize=10)
ax.set_xlabel('Click-Through Rate')
ax.set_ylabel('User Satisfaction')
ax.set_title('Reward Shaping: CTR vs Satisfaction Trade-off')
plt.tight_layout()
plt.show()

## 4. Delayed Rewards and Credit Assignment

In production, rewards often arrive with delay:
- **Immediate**: Clicks, impressions
- **Short-term** (minutes): Watch time, dwell time
- **Long-term** (days/weeks): Subscriptions, return visits, user retention

We need to assign credit from delayed rewards back to the actions that caused them.

$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots$$

With temporal difference learning, we bootstrap:

$$V(s_t) \leftarrow V(s_t) + \alpha [r_t + \gamma V(s_{t+1}) - V(s_t)]$$

In [ ]:
class DelayedRewardEnv:
    """Environment with delayed reward signals."""
    
    def __init__(self, n_items: int = 20, seed: int = 42):
        self.rng = np.random.RandomState(seed)
        self.n_items = n_items
        self.item_immediate_reward = self.rng.uniform(0, 1, n_items)
        self.item_delayed_reward = self.rng.uniform(0, 1, n_items)
        # Some items have high immediate but low delayed reward (clickbait)
        # and vice versa (quality content)
        self.item_immediate_reward[0:5] = np.array([0.9, 0.85, 0.8, 0.75, 0.7])  # Clickbait
        self.item_delayed_reward[0:5] = np.array([0.1, 0.15, 0.1, 0.2, 0.15])
        self.item_immediate_reward[5:10] = np.array([0.3, 0.35, 0.4, 0.25, 0.3])  # Quality
        self.item_delayed_reward[5:10] = np.array([0.8, 0.85, 0.9, 0.75, 0.8])
    
    def step(self, action: int) -> Tuple[float, float]:
        imm = float(self.rng.random() < self.item_immediate_reward[action])
        delayed = float(self.rng.random() < self.item_delayed_reward[action])
        return imm, delayed


def credit_assignment_experiment(env, gamma_values, n_rounds: int = 2000):
    """Show how discount factor affects which items are preferred."""
    results = {}
    
    for gamma in gamma_values:
        # Simple Q-learning
        q_values = np.zeros(env.n_items)
        counts = np.zeros(env.n_items)
        epsilon = 0.2
        
        for t in range(n_rounds):
            if np.random.random() < epsilon:
                action = np.random.randint(env.n_items)
            else:
                action = np.argmax(q_values)
            
            imm, delayed = env.step(action)
            reward = imm + gamma * delayed  # Weighted combination
            
            counts[action] += 1
            lr = 1.0 / (counts[action] + 1)
            q_values[action] += lr * (reward - q_values[action])
        
        best_item = np.argmax(q_values)
        item_type = "clickbait" if best_item < 5 else ("quality" if best_item < 10 else "other")
        results[gamma] = {'q_values': q_values.copy(), 'best_item': best_item, 'type': item_type}
        print(f"Gamma={gamma:.1f} | Best item: {best_item} ({item_type}) | Q-value: {q_values[best_item]:.3f}")
    
    return results


delay_env = DelayedRewardEnv(seed=42)
gamma_results = credit_assignment_experiment(delay_env, [0.0, 0.3, 0.5, 0.8, 1.0])

# Visualize Q-values for different gammas
fig, axes = plt.subplots(1, len(gamma_results), figsize=(16, 4), sharey=True)

for ax, (gamma, data) in zip(axes, gamma_results.items()):
    colors = ['red' if i < 5 else ('green' if i < 10 else 'gray') for i in range(delay_env.n_items)]
    ax.bar(range(delay_env.n_items), data['q_values'], color=colors, alpha=0.7)
    ax.set_title(f'gamma={gamma:.1f}\nBest: item {data["best_item"]} ({data["type"]})', fontsize=9)
    ax.set_xlabel('Item')
    if ax == axes[0]:
        ax.set_ylabel('Q-value')

# Legend
from matplotlib.patches import Patch
axes[-1].legend(handles=[
    Patch(facecolor='red', alpha=0.7, label='Clickbait'),
    Patch(facecolor='green', alpha=0.7, label='Quality'),
    Patch(facecolor='gray', alpha=0.7, label='Other')
], loc='upper right', fontsize=7)

plt.suptitle('Effect of Discount Factor on Item Preference', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Batch RL for Recommendation

In production, we often cannot interact with the environment in real-time. **Batch RL** (also called offline RL) learns entirely from a fixed dataset of logged interactions.

Key challenge: **distributional shift** — the learned policy may select actions not well-covered by the logging data.

Solutions:
- **Conservative Q-Learning (CQL)** (Kumar et al., 2020): Penalizes Q-values for out-of-distribution actions
- **Batch-Constrained Q-learning (BCQ)** (Fujimoto et al., 2019): Constrains actions to be close to the data

$$\mathcal{L}_{CQL} = \alpha \cdot \mathbb{E}_{s}\left[\log \sum_a \exp(Q(s,a)) - \mathbb{E}_{a \sim \mathcal{D}}[Q(s,a)]\right] + \mathcal{L}_{TD}$$

In [ ]:
class ConservativeQLearning:
    """Simplified Conservative Q-Learning for recommendation."""
    
    def __init__(self, state_dim: int, n_items: int, hidden_dim: int = 64,
                 cql_alpha: float = 1.0, lr: float = 1e-3):
        self.n_items = n_items
        self.cql_alpha = cql_alpha
        
        self.q_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_items)
        )
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
    
    def train_batch(self, states, actions, rewards, next_states, dones, gamma=0.99):
        """Train on a batch with CQL regularization."""
        states_t = torch.FloatTensor(states)
        actions_t = torch.LongTensor(actions)
        rewards_t = torch.FloatTensor(rewards)
        next_states_t = torch.FloatTensor(next_states)
        dones_t = torch.FloatTensor(dones)
        
        # Standard TD loss
        q_values = self.q_net(states_t)
        q_selected = q_values.gather(1, actions_t.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_q = self.q_net(next_states_t).max(1)[0]
            target = rewards_t + gamma * next_q * (1 - dones_t)
        
        td_loss = F.mse_loss(q_selected, target)
        
        # CQL regularization: penalize high Q-values for unseen actions
        logsumexp = torch.logsumexp(q_values, dim=1).mean()
        data_q = q_selected.mean()
        cql_loss = self.cql_alpha * (logsumexp - data_q)
        
        total_loss = td_loss + cql_loss
        
        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()
        
        return {'td_loss': td_loss.item(), 'cql_loss': cql_loss.item(), 'total': total_loss.item()}
    
    def select_action(self, state: np.ndarray) -> int:
        with torch.no_grad():
            q_values = self.q_net(torch.FloatTensor(state))
        return int(q_values.argmax())


# Prepare batch data
batch_states = np.array([d['state'] for d in logged_data])
batch_actions = np.array([d['action'] for d in logged_data])
batch_rewards = np.array([d['rewards']['satisfaction'] for d in logged_data])
# Use next state from subsequent entries (approximate)
batch_next_states = np.roll(batch_states, -1, axis=0)
batch_dones = np.zeros(len(logged_data))
batch_dones[9::10] = 1.0  # Episode boundaries

# Train CQL with different alpha values
cql_results = {}
for alpha in [0.0, 0.5, 1.0, 5.0]:
    cql = ConservativeQLearning(state_dim, env.n_items, cql_alpha=alpha)
    losses = []
    
    for epoch in range(20):
        idx = np.random.permutation(len(logged_data))
        epoch_loss = 0
        for i in range(0, len(idx), 128):
            batch_idx = idx[i:i+128]
            info = cql.train_batch(
                batch_states[batch_idx], batch_actions[batch_idx],
                batch_rewards[batch_idx], batch_next_states[batch_idx],
                batch_dones[batch_idx]
            )
            epoch_loss += info['total']
        losses.append(epoch_loss)
    
    # Evaluate
    def policy_fn(state, agent=cql):
        return agent.select_action(state)
    
    metrics = evaluate_policy(env, policy_fn, n_episodes=100)
    cql_results[alpha] = metrics
    print(f"CQL alpha={alpha:.1f} | CTR: {metrics['click_rate']:.4f} | Satisfaction: {metrics['avg_satisfaction']:.4f}")

## 6. Summary

| Production Concern | Solution | Key Insight |
|-------------------|----------|-------------|
| Off-policy learning | REINFORCE + IS correction | Top-K clipping for stability |
| Safe exploration | Constrained policies | Quality guardrails essential |
| Multiple objectives | Reward shaping | Balance short-term and long-term |
| Delayed rewards | Temporal credit assignment | Higher gamma favors quality |
| Logged data only | Batch RL (CQL) | Conservative Q-values reduce OOD risk |

**Key references:**
- Chen et al. (2019) - YouTube's REINFORCE system (Google)
- Kumar et al. (2020) - Conservative Q-Learning (CQL)
- Fujimoto et al. (2019) - Batch-Constrained Q-learning (BCQ)
- Thomas et al. (2015) - High-confidence off-policy evaluation

---

## Exercises

### 🏋️ Exercise 1: Off-Policy Corrected REINFORCE for Ranking

Implement the full YouTube-style off-policy corrected REINFORCE with top-K correction and a variance-reducing baseline.

In [ ]:
# 🏋️ Exercise 1: Full off-policy REINFORCE with baseline

class OffPolicyREINFORCEWithBaseline(nn.Module):
    """Off-policy REINFORCE with learned value baseline."""
    
    def __init__(self, state_dim: int, n_items: int, hidden_dim: int = 64):
        super().__init__()
        # TODO: Define policy network (state -> item probabilities)
        self.policy_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_items)
        )
        # TODO: Define value baseline network (state -> scalar)
        self.value_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def get_policy(self, state: torch.Tensor) -> torch.Tensor:
        # TODO: Return softmax probabilities
        pass
    
    def get_value(self, state: torch.Tensor) -> torch.Tensor:
        # TODO: Return value estimate
        pass


def train_with_baseline(logged_data, agent, optimizer, n_epochs=20, top_k=10):
    # TODO: Train with off-policy correction AND value baseline
    # Loss = policy_loss + 0.5 * value_loss
    # policy_loss uses (reward - value) as advantage
    pass

print("Exercise 1: Implement OffPolicyREINFORCEWithBaseline")

### 🏋️ Exercise 2: Adaptive Reward Shaping

Implement a reward shaping strategy that adapts weights based on the current state of the recommendation system.

In [ ]:
# 🏋️ Exercise 2: Adaptive Reward Shaping

class AdaptiveRewardShaper:
    """Adapts reward weights based on running metrics."""
    
    def __init__(self, target_ctr: float = 0.3, target_satisfaction: float = 0.2):
        self.target_ctr = target_ctr
        self.target_satisfaction = target_satisfaction
        self.running_ctr = 0.0
        self.running_satisfaction = 0.0
        self.alpha = 0.01  # EMA decay
    
    def shape(self, rewards: Dict[str, float]) -> float:
        # TODO: Implement adaptive weighting:
        # - If CTR is below target, increase click weight
        # - If satisfaction is below target, increase satisfaction weight
        # - Update running averages
        pass

print("Exercise 2: Implement AdaptiveRewardShaper")

### 🏋️ Exercise 3: A/B Test Simulation

Simulate an A/B test between two RL policies and determine the minimum sample size needed for significance.

In [ ]:
# 🏋️ Exercise 3: A/B Test Simulation

def simulate_ab_test(env, policy_a_fn, policy_b_fn, 
                     n_users_per_group: int = 1000, n_recs_per_user: int = 10):
    """Simulate A/B test between two policies."""
    # TODO: 
    # 1. Run both policies on separate user groups
    # 2. Collect click rates and satisfaction for each group
    # 3. Compute t-statistic and p-value
    # 4. Return whether the difference is statistically significant
    pass

# TODO: Run the simulation with varying group sizes to find minimum sample size
# for detecting a 5% improvement in satisfaction

print("Exercise 3: Implement A/B test simulation")